In [ ]:
# Scrapa detaljsidor från Hemnet
# ============================================================
# STEG 1: Importera paket
# ============================================================
import pandas as pd
import numpy as np
import time
import random
import re
import json
import os
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
from tqdm.notebook import tqdm

print('Paket laddade!')

Paket laddade!


In [2]:
# ============================================================
# STEG 2: Ladda befintlig data (som har URL:er)
# ============================================================
df = pd.read_csv('../data/processed/orebro_housing_clean.csv')
print(f'Laddad: {len(df)} rader')

# Hämta unika URL:er
urls = df['url'].dropna().unique().tolist()
print(f'Unika URL:er att scrapa: {len(urls)}')
print(f'Exempel: {urls[0]}')

Laddad: 6600 rader
Unika URL:er att scrapa: 6600
Exempel: https://www.hemnet.se/salda/lagenhet-4rum-rynninge-orebro-kommun-kaserngarden-14-7127558886027008187


In [3]:
# ============================================================
# STEG 3: Selenium-driver
# ============================================================
def create_driver():
    options = Options()
    options.add_argument('--headless=new')  # Kommentera bort för att se webbläsaren
    options.add_argument('--no-sandbox')
    options.add_argument('--disable-dev-shm-usage')
    options.add_argument('--window-size=1920,1080')
    options.add_argument('user-agent=Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) '
                         'AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36')
    service = Service(ChromeDriverManager().install())
    return webdriver.Chrome(service=service, options=options)

# Testa
driver = create_driver()
driver.get('https://www.hemnet.se')
print(f'Selenium fungerar! Titel: {driver.title}')
driver.quit()

Selenium fungerar! Titel: Hemnet - Sveriges största bostadsplattform


In [4]:
# ============================================================
# STEG 4: Parser för detaljsidor (v2 — testad och fungerar)
# ============================================================
#
# Hemnet visar INTE <dt>/<dd> på sålda bostäder.
# Istället ligger datan som text-rader:
#   Byggår
#   2006
#   Våning
#   5 av 6, hiss finns
#   Driftskostnad
#   8 400 kr/år
#
# Parsern läser alla rader och matchar etikett → värde på nästa rad.
# Testad på riktiga Hemnet-sidor 2026-03-16 — hittade 14 features.

def parse_detail_page(html):
    """Extrahera features från Hemnet-detaljsida via text-rader."""
    soup = BeautifulSoup(html, 'lxml')
    details = {}
    
    # Hämta all text som en lista av rader
    text = soup.get_text('\n', strip=True)
    lines = [line.strip() for line in text.split('\n') if line.strip()]
    
    # Gå igenom varje rad — om raden är en etikett, ta värdet från nästa rad
    for i, line in enumerate(lines):
        if i + 1 >= len(lines):
            break
        
        next_line = lines[i + 1]
        label = line.lower().strip()
        
        # --- Byggår ---
        if label in ['byggår', 'byggnadsår']:
            year_match = re.search(r'(1[6-9]\d{2}|20[0-2]\d)', next_line)
            if year_match:
                details['byggar'] = int(year_match.group(1))
        
        # --- Våning (+ hiss-info) ---
        elif label == 'våning':
            floor_match = re.search(r'(\d+)', next_line)
            if floor_match:
                details['vaning'] = int(floor_match.group(1))
            total_match = re.search(r'av\s*(\d+)', next_line)
            if total_match:
                details['antal_vaningar'] = int(total_match.group(1))
            # "5 av 6, hiss finns" → fånga hiss
            if 'hiss' in next_line.lower():
                details['har_hiss'] = 1
        
        # --- Balkong ---
        elif label == 'balkong':
            details['har_balkong'] = 1 if next_line.lower().strip() in ['ja', 'yes'] else 0
        
        # --- Boarea (backup) ---
        elif label == 'boarea':
            area_match = re.search(r'([\d\s,\.]+)\s*m', next_line)
            if area_match:
                details['boarea_detail'] = float(
                    area_match.group(1).replace(' ', '').replace(',', '.').replace('\xa0', ''))
        
        # --- Biarea ---
        elif label == 'biarea':
            area_match = re.search(r'([\d\s,\.]+)\s*m', next_line)
            if area_match:
                details['biarea_kvm'] = float(
                    area_match.group(1).replace(' ', '').replace(',', '.').replace('\xa0', ''))
        
        # --- Tomtarea ---
        elif label == 'tomtarea':
            area_match = re.search(r'([\d\s,\.]+)\s*m', next_line)
            if area_match:
                details['tomtarea_kvm'] = float(
                    area_match.group(1).replace(' ', '').replace(',', '.').replace('\xa0', ''))
        
        # --- Avgift ---
        elif label == 'avgift':
            fee_match = re.search(r'([\d\s]+)', next_line)
            if fee_match:
                details['avgift_detail'] = int(
                    fee_match.group(1).replace(' ', '').replace('\xa0', ''))
        
        # --- Driftskostnad ---
        elif label in ['driftskostnad', 'driftkostnad']:
            cost_match = re.search(r'([\d\s]+)', next_line)
            if cost_match:
                details['driftkostnad_ar'] = int(
                    cost_match.group(1).replace(' ', '').replace('\xa0', ''))
        
        # --- Antal rum ---
        elif label == 'antal rum':
            room_match = re.search(r'([\d,]+)', next_line)
            if room_match:
                details['antal_rum_detail'] = float(room_match.group(1).replace(',', '.'))
        
        # --- Förening ---
        elif label == 'förening':
            details['forening'] = next_line
        
        # --- Upplåtelseform ---
        elif label == 'upplåtelseform':
            details['upplatelseform'] = next_line
        
        # --- Antal besök (popularitet!) ---
        elif label == 'antal besök':
            visit_match = re.search(r'([\d\s]+)', next_line)
            if visit_match:
                details['antal_besok'] = int(
                    visit_match.group(1).replace(' ', '').replace('\xa0', ''))
    
    # --- Komplettera med textsökning för binära features ---
    full_text = text.lower()
    
    if 'har_hiss' not in details:
        details['har_hiss'] = 1 if re.search(r'\bhiss\b', full_text) else 0
    
    if 'har_balkong' not in details:
        details['har_balkong'] = 1 if re.search(r'\bbalkong\b', full_text) else 0
    
    details['har_uteplats'] = 1 if re.search(r'\b(uteplats|terrass|altan)\b', full_text) else 0
    details['har_garage'] = 1 if re.search(r'\b(garage|carport)\b', full_text) else 0
    details['renoverad'] = 1 if re.search(r'\b(renovera|totalrenover|nyrenovera|stamby)\b', full_text) else 0
    
    return details

print('Parser v2 redo!')
print('Fångar: byggår, våning, antal_vaningar, hiss, balkong, biarea, tomtarea,')
print('        avgift, driftkostnad, antal_rum, förening, upplåtelseform,')
print('        antal_besök, uteplats, garage, renoverad')

Parser v2 redo!
Fångar: byggår, våning, antal_vaningar, hiss, balkong, biarea, tomtarea,
        avgift, driftkostnad, antal_rum, förening, upplåtelseform,
        antal_besök, uteplats, garage, renoverad


In [5]:
# ============================================================
# STEG 5: Testa parsern på EN sida
# ============================================================
test_url = urls[0]
print(f'Testar URL: {test_url}')

driver = create_driver()
driver.get(test_url)
time.sleep(3)
result = parse_detail_page(driver.page_source)
driver.quit()

print(f'\nHittade {len(result)} features:')
for key, val in sorted(result.items()):
    print(f'  {key}: {val}')

Testar URL: https://www.hemnet.se/salda/lagenhet-4rum-rynninge-orebro-kommun-kaserngarden-14-7127558886027008187

Hittade 14 features:
  antal_besok: 7251
  antal_rum_detail: 4.0
  antal_vaningar: 6
  avgift_detail: 8768
  boarea_detail: 117.0
  byggar: 2006
  driftkostnad_ar: 8400
  har_balkong: 0
  har_garage: 0
  har_hiss: 1
  har_uteplats: 1
  renoverad: 0
  upplatelseform: Bostadsrätt
  vaning: 5


In [6]:
# ============================================================
# STEG 5b: Testa på 3 olika URL:er (lägenhet, villa, radhus)
# ============================================================

test_indices = [0, len(urls)//3, 2*len(urls)//3]

driver = create_driver()
for idx in test_indices:
    url = urls[idx]
    driver.get(url)
    time.sleep(3)
    result = parse_detail_page(driver.page_source)
    print(f'URL {idx}: {len(result)} features')
    for key, val in sorted(result.items()):
        print(f'  {key}: {val}')
    print()
driver.quit()

print('Om alla 3 visar features → redo att köra steg 6!')

URL 0: 14 features
  antal_besok: 7251
  antal_rum_detail: 4.0
  antal_vaningar: 6
  avgift_detail: 8768
  boarea_detail: 117.0
  byggar: 2006
  driftkostnad_ar: 8400
  har_balkong: 0
  har_garage: 0
  har_hiss: 1
  har_uteplats: 1
  renoverad: 0
  upplatelseform: Bostadsrätt
  vaning: 5

URL 2200: 12 features
  antal_besok: 2232
  antal_rum_detail: 4.0
  avgift_detail: 5588
  boarea_detail: 95.8
  byggar: 2016
  har_balkong: 0
  har_garage: 0
  har_hiss: 1
  har_uteplats: 1
  renoverad: 0
  upplatelseform: Bostadsrätt
  vaning: 3

URL 4400: 13 features
  antal_besok: 2516
  antal_rum_detail: 4.0
  biarea_kvm: 124.0
  boarea_detail: 124.0
  byggar: 1971
  driftkostnad_ar: 29000
  har_balkong: 0
  har_garage: 0
  har_hiss: 0
  har_uteplats: 1
  renoverad: 0
  tomtarea_kvm: 391.0
  upplatelseform: Äganderätt

Om alla 3 visar features → redo att köra steg 6!


In [7]:
# ============================================================
# STEG 6: SCRAPA ALLA DETALJSIDOR
# ============================================================
#
# ~6600 sidor → 3-5 timmar med 2-4 sek delay per sida.
#
# FUNKTIONER:
#   ✅ Sparar automatiskt efter varje 50:e sida
#   ✅ Om den kraschar: kör cellen igen → fortsätter där den slutade
#   ✅ Startar om Selenium var 200:e sida (förebygger problem)
#   ✅ Loggar framsteg var 100:e sida
#
# TIPS: Stäng av viloläge i Systeminställningar → Energi
#
# VILL DU TESTA FÖRST? Byt remaining_urls → remaining_urls[:20]

SAVE_PATH = '../data/raw/hemnet_details.json'
CHECKPOINT_EVERY = 50
RESTART_DRIVER_EVERY = 200

# Ladda redan scrapade resultat (om de finns)
if os.path.exists(SAVE_PATH):
    with open(SAVE_PATH, 'r', encoding='utf-8') as f:
        all_details = json.load(f)
    already_scraped = set(d['url'] for d in all_details)
    print(f'Laddat {len(all_details)} redan scrapade sidor')
else:
    all_details = []
    already_scraped = set()
    print('Börjar från noll')

# Filtrera bort redan scrapade
remaining_urls = [u for u in urls if u not in already_scraped]
print(f'Kvar att scrapa: {len(remaining_urls)} av {len(urls)}')

if len(remaining_urls) == 0:
    print('\nAllt redan scrapat! Hoppa till steg 7.')
else:
    driver = create_driver()
    errors = []
    
    try:
        for i, url in enumerate(tqdm(remaining_urls, desc='Scrapar detaljsidor')):
            try:
                driver.get(url)
                time.sleep(random.uniform(2, 4))
                
                details = parse_detail_page(driver.page_source)
                details['url'] = url
                all_details.append(details)
                
            except Exception as e:
                errors.append({'url': url, 'error': str(e)})
            
            # Checkpoint-sparning
            if (i + 1) % CHECKPOINT_EVERY == 0:
                with open(SAVE_PATH, 'w', encoding='utf-8') as f:
                    json.dump(all_details, f, ensure_ascii=False)
            
            # Starta om drivern regelbundet
            if (i + 1) % RESTART_DRIVER_EVERY == 0:
                driver.quit()
                time.sleep(random.uniform(5, 10))
                driver = create_driver()
                print(f'  Driver omstartad vid sida {i+1}')
            
            # Logga framsteg
            if (i + 1) % 100 == 0:
                success_rate = (len(all_details) / (len(all_details) + len(errors))) * 100
                print(f'  {i+1}/{len(remaining_urls)} — '
                      f'{len(all_details)} OK, {len(errors)} fel ({success_rate:.1f}% success)')
    
    finally:
        driver.quit()
        # Slutgiltig sparning (körs ALLTID, även vid krasch)
        with open(SAVE_PATH, 'w', encoding='utf-8') as f:
            json.dump(all_details, f, ensure_ascii=False)
        print(f'\nSparad! {len(all_details)} sidor totalt')
    
    print(f'\nKlart!')
    print(f'  Scrapade: {len(all_details)}')
    print(f'  Fel: {len(errors)}')
    print(f'  Sparad till: {SAVE_PATH}')

Laddat 0 redan scrapade sidor
Kvar att scrapa: 6600 av 6600


Scrapar detaljsidor:   0%|          | 0/6600 [00:00<?, ?it/s]

  100/6600 — 100 OK, 0 fel (100.0% success)
  Driver omstartad vid sida 200
  200/6600 — 200 OK, 0 fel (100.0% success)
  300/6600 — 300 OK, 0 fel (100.0% success)
  Driver omstartad vid sida 400
  400/6600 — 400 OK, 0 fel (100.0% success)
  500/6600 — 500 OK, 0 fel (100.0% success)
  Driver omstartad vid sida 600
  600/6600 — 600 OK, 0 fel (100.0% success)
  700/6600 — 700 OK, 0 fel (100.0% success)
  Driver omstartad vid sida 800
  800/6600 — 800 OK, 0 fel (100.0% success)
  900/6600 — 900 OK, 0 fel (100.0% success)
  Driver omstartad vid sida 1000
  1000/6600 — 1000 OK, 0 fel (100.0% success)
  1100/6600 — 1100 OK, 0 fel (100.0% success)
  Driver omstartad vid sida 1200
  1200/6600 — 1200 OK, 0 fel (100.0% success)
  1300/6600 — 1300 OK, 0 fel (100.0% success)
  Driver omstartad vid sida 1400
  1400/6600 — 1400 OK, 0 fel (100.0% success)
  1500/6600 — 1500 OK, 0 fel (100.0% success)
  Driver omstartad vid sida 1600
  1600/6600 — 1600 OK, 0 fel (100.0% success)
  1700/6600 — 1700 OK,

In [8]:
# ============================================================
# STEG 7: Granska resultaten
# ============================================================

with open(SAVE_PATH, 'r', encoding='utf-8') as f:
    all_details = json.load(f)

df_details = pd.DataFrame(all_details)
print(f'Totalt: {len(df_details)} rader')
print(f'Kolumner: {list(df_details.columns)}')

print(f'\n=== Täckningsgrad ===')
for col in sorted(df_details.columns):
    if col != 'url':
        count = df_details[col].notna().sum()
        pct = count / len(df_details) * 100
        print(f'  {col:25s}: {count:5d} ({pct:5.1f}%)')

print(f'\n=== Statistik ===')
numeric_cols = ['byggar', 'vaning', 'antal_vaningar', 'tomtarea_kvm', 
                'biarea_kvm', 'driftkostnad_ar', 'antal_besok']
for col in numeric_cols:
    if col in df_details.columns:
        s = df_details[col].dropna()
        if len(s) > 0:
            print(f'  {col:25s}: min={s.min():.0f}, median={s.median():.0f}, max={s.max():.0f}')

Totalt: 6600 rader
Kolumner: ['upplatelseform', 'antal_rum_detail', 'boarea_detail', 'har_balkong', 'vaning', 'antal_vaningar', 'har_hiss', 'byggar', 'avgift_detail', 'driftkostnad_ar', 'antal_besok', 'har_uteplats', 'har_garage', 'renoverad', 'url', 'biarea_kvm', 'tomtarea_kvm']

=== Täckningsgrad ===
  antal_besok              :  6526 ( 98.9%)
  antal_rum_detail         :  6496 ( 98.4%)
  antal_vaningar           :  2267 ( 34.3%)
  avgift_detail            :  3735 ( 56.6%)
  biarea_kvm               :  1850 ( 28.0%)
  boarea_detail            :  6587 ( 99.8%)
  byggar                   :  5997 ( 90.9%)
  driftkostnad_ar          :  5468 ( 82.8%)
  har_balkong              :  6600 (100.0%)
  har_garage               :  6600 (100.0%)
  har_hiss                 :  6600 (100.0%)
  har_uteplats             :  6600 (100.0%)
  renoverad                :  6600 (100.0%)
  tomtarea_kvm             :  2748 ( 41.6%)
  upplatelseform           :  6596 ( 99.9%)
  vaning                   :  2443 (

In [9]:
# ============================================================
# STEG 8: Koppla ihop med huvuddatan
# ============================================================

df_main = pd.read_csv('../data/processed/orebro_housing_clean.csv')
print(f'Huvuddata: {len(df_main)} rader')

df_det = pd.DataFrame(all_details)
print(f'Detaljdata: {len(df_det)} rader')

# Ta bort kolumner som redan finns i huvuddatan
drop_cols = ['boarea_detail', 'antal_rum_detail', 'avgift_detail']
for col in drop_cols:
    if col in df_det.columns:
        df_det = df_det.drop(col, axis=1)

# Merga på URL
df_merged = df_main.merge(df_det, on='url', how='left')
print(f'\nMergad: {len(df_merged)} rader')

# Kolla matchning
for col in ['byggar', 'vaning', 'driftkostnad_ar', 'har_hiss', 'har_balkong', 'antal_besok']:
    if col in df_merged.columns:
        n = df_merged[col].notna().sum()
        print(f'  {col:25s}: {n:5d} matchade ({n/len(df_merged)*100:.1f}%)')

Huvuddata: 6600 rader
Detaljdata: 6600 rader

Mergad: 6600 rader
  byggar                   :  5997 matchade (90.9%)
  vaning                   :  2443 matchade (37.0%)
  driftkostnad_ar          :  5468 matchade (82.8%)
  har_hiss                 :  6600 matchade (100.0%)
  har_balkong              :  6600 matchade (100.0%)
  antal_besok              :  6526 matchade (98.9%)


In [10]:
# ============================================================
# STEG 9: Feature engineering med nya features
# ============================================================

df = df_merged.copy()

# --- Bostadens ålder ---
if 'byggar' in df.columns:
    df['bostad_alder'] = 2026 - df['byggar']
    df.loc[df['bostad_alder'] < 0, 'bostad_alder'] = np.nan
    df.loc[df['bostad_alder'] > 300, 'bostad_alder'] = np.nan
    print(f'bostad_alder: median={df["bostad_alder"].median():.0f} år, '
          f'saknas={df["bostad_alder"].isna().sum()}')

# --- Total yta (boarea + biarea) ---
if 'biarea_kvm' in df.columns:
    df['total_yta'] = df['boarea_kvm'] + df['biarea_kvm'].fillna(0)
    print(f'total_yta: median={df["total_yta"].median():.0f} m²')

# --- Relativ våning ---
if 'vaning' in df.columns and 'antal_vaningar' in df.columns:
    mask = df['antal_vaningar'].notna() & (df['antal_vaningar'] > 0)
    df.loc[mask, 'relativ_vaning'] = df.loc[mask, 'vaning'] / df.loc[mask, 'antal_vaningar']
    df['toppvaning'] = (df['vaning'] == df['antal_vaningar']).astype(int)
    df.loc[df['vaning'].isna(), 'toppvaning'] = np.nan
    print(f'toppvaning: {df["toppvaning"].sum():.0f} st')

# --- Driftkostnad per kvm ---
if 'driftkostnad_ar' in df.columns:
    mask = df['driftkostnad_ar'].notna() & (df['boarea_kvm'] > 0)
    df.loc[mask, 'driftkostnad_per_kvm'] = df.loc[mask, 'driftkostnad_ar'] / df.loc[mask, 'boarea_kvm']
    print(f'driftkostnad_per_kvm: median={df["driftkostnad_per_kvm"].median():.0f} kr/kvm/år')

# --- Tomtarea per boarea ---
if 'tomtarea_kvm' in df.columns:
    mask = df['tomtarea_kvm'].notna() & (df['boarea_kvm'] > 0)
    df.loc[mask, 'tomt_per_boarea'] = df.loc[mask, 'tomtarea_kvm'] / df.loc[mask, 'boarea_kvm']
    print(f'tomt_per_boarea: median={df["tomt_per_boarea"].median():.1f}')

# --- Fyll binära features med 0 ---
for col in ['har_hiss', 'har_balkong', 'har_uteplats', 'har_garage', 'renoverad']:
    if col in df.columns:
        df[col] = df[col].fillna(0).astype(int)

# --- Fyll bostad_alder med median ---
if 'bostad_alder' in df.columns:
    median_alder = df['bostad_alder'].median()
    saknade = df['bostad_alder'].isna().sum()
    df['bostad_alder'] = df['bostad_alder'].fillna(median_alder)
    print(f'bostad_alder: fyllde {saknade} saknade med median={median_alder:.0f}')

print(f'\n=== Sammanfattning ===')
new_cols = ['bostad_alder', 'total_yta', 'vaning', 'relativ_vaning', 'toppvaning',
            'driftkostnad_ar', 'driftkostnad_per_kvm', 'tomtarea_kvm', 'tomt_per_boarea',
            'biarea_kvm', 'antal_besok', 'har_hiss', 'har_balkong', 'har_uteplats',
            'har_garage', 'renoverad']
for col in new_cols:
    if col in df.columns:
        non_null = df[col].notna().sum()
        print(f'  {col:25s}: {non_null:5d} ({non_null/len(df)*100:.1f}%)')

bostad_alder: median=48 år, saknas=607
total_yta: median=107 m²
toppvaning: 737 st
driftkostnad_per_kvm: median=189 kr/kvm/år
tomt_per_boarea: median=6.6
bostad_alder: fyllde 607 saknade med median=48

=== Sammanfattning ===
  bostad_alder             :  6600 (100.0%)
  total_yta                :  6595 (99.9%)
  vaning                   :  2443 (37.0%)
  relativ_vaning           :  2267 (34.3%)
  toppvaning               :  2443 (37.0%)
  driftkostnad_ar          :  5468 (82.8%)
  driftkostnad_per_kvm     :  5466 (82.8%)
  tomtarea_kvm             :  2748 (41.6%)
  tomt_per_boarea          :  2748 (41.6%)
  biarea_kvm               :  1850 (28.0%)
  antal_besok              :  6526 (98.9%)
  har_hiss                 :  6600 (100.0%)
  har_balkong              :  6600 (100.0%)
  har_uteplats             :  6600 (100.0%)
  har_garage               :  6600 (100.0%)
  renoverad                :  6600 (100.0%)


In [11]:
# ============================================================
# STEG 10: Korrelationscheck — vilka nya features påverkar priset?
# ============================================================

check_cols = ['bostad_alder', 'vaning', 'toppvaning', 'driftkostnad_ar',
              'tomtarea_kvm', 'biarea_kvm', 'antal_besok',
              'har_hiss', 'har_balkong', 'har_uteplats', 'har_garage', 'renoverad']

print('Korrelation med slutpris (nya features):')
print('=' * 55)
corrs = []
for col in check_cols:
    if col in df.columns:
        c = df[[col, 'slutpris']].dropna().corr().iloc[0, 1]
        corrs.append((col, c))

corrs.sort(key=lambda x: abs(x[1]), reverse=True)
for col, c in corrs:
    bar = '█' * int(abs(c) * 40)
    sign = '+' if c > 0 else '-'
    print(f'  {col:25s}: {sign}{abs(c):.3f}  {bar}')

Korrelation med slutpris (nya features):
  driftkostnad_ar          : +0.621  ████████████████████████
  har_hiss                 : -0.463  ██████████████████
  antal_besok              : +0.396  ███████████████
  vaning                   : +0.204  ████████
  biarea_kvm               : +0.195  ███████
  tomtarea_kvm             : -0.174  ██████
  har_uteplats             : -0.124  ████
  bostad_alder             : +0.090  ███
  toppvaning               : -0.039  █


ValueError: cannot convert float NaN to integer

In [13]:
# ============================================================
# STEG 11: Spara den berikade datan
# ============================================================

output_path = '../data/processed/orebro_housing_enriched.csv'
df.to_csv(output_path, index=False, encoding='utf-8-sig')

print(f'Sparad: {output_path}')
print(f'Rader: {len(df)}')
print(f'Kolumner: {len(df.columns)}')

print(f'\nNya kolumner:')
old_cols = set(df_main.columns)
for col in sorted(df.columns):
    if col not in old_cols:
        print(f'  + {col}')

print(f'\n{"="*50}')
print(f'KLART! Nästa steg:')
print(f'1. Öppna 03_modeling.ipynb')
print(f'2. Ändra: pd.read_csv("../data/processed/orebro_housing_enriched.csv")')
print(f'{"="*50}')

Sparad: ../data/processed/orebro_housing_enriched.csv
Rader: 6600
Kolumner: 41

Nya kolumner:
  + antal_besok
  + antal_vaningar
  + biarea_kvm
  + bostad_alder
  + byggar
  + driftkostnad_ar
  + driftkostnad_per_kvm
  + har_balkong
  + har_garage
  + har_hiss
  + har_uteplats
  + relativ_vaning
  + renoverad
  + tomt_per_boarea
  + tomtarea_kvm
  + toppvaning
  + total_yta
  + upplatelseform
  + vaning

KLART! Nästa steg:
1. Öppna 03_modeling.ipynb
2. Ändra: pd.read_csv("../data/processed/orebro_housing_enriched.csv")
